# VBZ Geo-Map — Plotly

Interaktive Kartenvisualisierung mit **Plotly / Maplibre**.

---

## Architektur-Notizen

**Stärken:**
- Direkt in VS Code renderbar, kein Browser nötig
- Dashboard-fähig via Plotly Dash
- Hover, Zoom, Pan out of the box

**Einschränkung Segment-Einfärbung:**
- Plotly braucht einen eigenen Trace pro Farbsegment
- Alle Linien + alle Segmente (~500 Traces) crasht den Browser-Renderer
- **Lösung:** Immer gefiltert — eine Linie per Dropdown, oder Top-N Segmente
- Eine einzelne Linie hat ~10-30 Segmente → problemlos
- Für Gesamtnetz-Übersicht aller Linien gleichzeitig → Folium oder Kepler.gl


In [1]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'data' / 'interim').exists():
        _ROOT = _p
        break

GTFS_DIR = _ROOT / 'data' / 'interim' / 'gtfs'
GEO_DIR  = _ROOT / 'data' / 'raw' / 'geo' / 'data'

routes = pd.read_parquet(GTFS_DIR / 'gtfs_tram_routes.parquet')
stops  = pd.read_parquet(GTFS_DIR / 'gtfs_tram_stops.parquet')
shapes = pd.read_parquet(GTFS_DIR / 'gtfs_tram_shapes.parquet')
trips  = pd.read_parquet(GTFS_DIR / 'gtfs_tram_trips.parquet')

with open(GEO_DIR / 'stzh_adm_stadtkreise_v.json') as f:
    stadtkreise_geo = json.load(f)
with open(GEO_DIR / 'stzh_adm_stadtkreise_beschr_p.json') as f:
    labels_geo = json.load(f)

routes_2024  = routes[routes['year'] == '2024'].copy()
shapes_2024  = shapes[shapes['year'] == '2024'].copy()
stops_2024   = stops[stops['year'] == '2024'].copy()
trips_2024   = trips[trips['year'] == '2024'].copy()
stops_unique = stops_2024.drop_duplicates(subset=['stop_name']).copy()

np.random.seed(42)
stops_unique['delay_min'] = np.random.exponential(scale=1.5, size=len(stops_unique)).round(1)

line_colors = dict(zip(routes_2024['route_short_name'], '#' + routes_2024['route_color']))
kreis_map   = {f['properties']['objid']: f['properties']['kname']
               for f in stadtkreise_geo['features']}

def best_shape_for_line(line_name):
    route_ids = routes_2024[routes_2024['route_short_name'] == line_name]['route_id']
    shape_ids = trips_2024[trips_2024['route_id'].isin(route_ids)]['shape_id'].unique()
    if len(shape_ids) == 0:
        return None
    return max(shape_ids, key=lambda s: len(shapes_2024[shapes_2024['shape_id'] == s]))

print(f'Routes: {len(routes_2024)}  Stops: {len(stops_unique)}')


Routes: 16  Stops: 1189


## Karte 1 — Stadtkreise + Tramnetz + Haltestellen


In [2]:
fig = go.Figure()

for feature in stadtkreise_geo['features']:
    name   = feature['properties']['kname']
    coords = feature['geometry']['coordinates'][0]
    fig.add_trace(go.Scattermap(
        lat=[c[1] for c in coords], lon=[c[0] for c in coords],
        mode='lines',
        line=dict(color='rgba(255,255,255,0.35)', width=1),
        fill='toself', fillcolor='rgba(255,255,255,0.04)',
        hovertemplate=f'{name}<extra></extra>',
        showlegend=False
    ))

for feature in labels_geo['features']:
    lon, lat = feature['geometry']['coordinates']
    name = kreis_map.get(feature['properties']['objid'], '')
    fig.add_trace(go.Scattermap(
        lat=[lat], lon=[lon], mode='text', text=[name],
        textfont=dict(size=11, color='rgba(255,255,255,0.6)'),
        showlegend=False
    ))

for line in sorted(routes_2024['route_short_name'].unique()):
    shape_id = best_shape_for_line(line)
    if shape_id is None:
        continue
    pts = (shapes_2024[shapes_2024['shape_id'] == shape_id]
           .sort_values('shape_pt_sequence').iloc[::5])
    fig.add_trace(go.Scattermap(
        lat=pts['shape_pt_lat'].tolist(), lon=pts['shape_pt_lon'].tolist(),
        mode='lines', line=dict(color=line_colors.get(line, '#999'), width=2.5),
        name=f'Linie {line}', hoverinfo='name'
    ))

fig.add_trace(go.Scattermap(
    lat=stops_unique['stop_lat'].tolist(), lon=stops_unique['stop_lon'].tolist(),
    mode='markers', marker=dict(size=5, color='white', opacity=0.7),
    name='Haltestellen', text=stops_unique['stop_name'].tolist(),
    hovertemplate='%{text}<extra></extra>'
))

fig.update_layout(
    map=dict(style='dark', center=dict(lat=47.378, lon=8.540), zoom=12),
    margin=dict(l=0, r=0, t=35, b=0), height=650,
    title='Tramnetz Zürich 2024',
    legend=dict(bgcolor='rgba(0,0,0,0.5)', font=dict(color='white', size=11))
)
fig.show()


## Karte 2 — Verspätungs-Heatmap


In [3]:
fig2 = px.density_map(
    stops_unique, lat='stop_lat', lon='stop_lon', z='delay_min',
    radius=22, center=dict(lat=47.378, lon=8.540), zoom=12,
    map_style='dark', color_continuous_scale='Inferno',
    range_color=[0, stops_unique['delay_min'].quantile(0.95)],
    title='Verspätungs-Heatmap (synthetische Daten)',
    labels={'delay_min': 'Verspätung (Min)'}
)
fig2.update_layout(height=650, margin=dict(l=0, r=0, t=35, b=0))
fig2.show()


## Karte 2b — Haltestellen eingefärbt nach Verspätung


In [4]:
max_delay = stops_unique['delay_min'].quantile(0.95)

fig2b = go.Figure()

for feature in stadtkreise_geo['features']:
    coords = feature['geometry']['coordinates'][0]
    fig2b.add_trace(go.Scattermap(
        lat=[c[1] for c in coords], lon=[c[0] for c in coords],
        mode='lines', line=dict(color='rgba(255,255,255,0.2)', width=1),
        showlegend=False
    ))

fig2b.add_trace(go.Scattermap(
    lat=stops_unique['stop_lat'].tolist(), lon=stops_unique['stop_lon'].tolist(),
    mode='markers',
    marker=dict(
        size=10, color=stops_unique['delay_min'].tolist(),
        colorscale='RdYlGn_r',
        colorbar=dict(title='Verspätung (Min)', thickness=15),
        cmin=0, cmax=max_delay, opacity=0.85
    ),
    text=stops_unique['stop_name'].tolist(),
    customdata=stops_unique['delay_min'].tolist(),
    hovertemplate='<b>%{text}</b><br>%{customdata:.1f} Min<extra></extra>',
    name='Verspätung'
))

fig2b.update_layout(
    map=dict(style='dark', center=dict(lat=47.378, lon=8.540), zoom=12),
    margin=dict(l=0, r=0, t=35, b=0), height=650,
    title='Verspätung pro Haltestelle (synthetische Daten)'
)
fig2b.show()


## Karte 3 — Verspätung auf Streckenabschnitten (Linie 11)

> Eine Linie pro Darstellung (~20 Traces). Im Dashboard via Dropdown wählbar.


In [5]:
# Drei Haltestellen auf Linie 11
h1 = stops_2024[stops_2024['stop_name'] == 'Zürich, Paradeplatz'].iloc[0]
h2 = stops_2024[stops_2024['stop_name'] == 'Zürich, Bellevue'].iloc[0]
h3 = stops_2024[stops_2024['stop_name'] == 'Zürich, Bahnhof Stadelhofen'].iloc[0]

route_ids_11 = routes_2024[routes_2024['route_short_name'] == '11']['route_id']
shape_ids_11 = trips_2024[trips_2024['route_id'].isin(route_ids_11)]['shape_id'].unique()
shape_id_11  = max(shape_ids_11, key=lambda s: len(shapes_2024[shapes_2024['shape_id'] == s]))
shape_11     = shapes_2024[shapes_2024['shape_id'] == shape_id_11].sort_values('shape_pt_sequence')
color_11     = '#' + routes_2024[routes_2024['route_short_name'] == '11']['route_color'].iloc[0]

fig3 = go.Figure()

# Stadtkreise
for feature in stadtkreise_geo['features']:
    coords = feature['geometry']['coordinates'][0]
    fig3.add_trace(go.Scattermap(
        lat=[c[1] for c in coords], lon=[c[0] for c in coords],
        mode='lines', line=dict(color='rgba(255,255,255,0.15)', width=1),
        showlegend=False
    ))

# Linie 11 in VBZ-Farbe
fig3.add_trace(go.Scattermap(
    lat=shape_11['shape_pt_lat'].tolist(), lon=shape_11['shape_pt_lon'].tolist(),
    mode='lines', line=dict(color=color_11, width=3),
    name='Linie 11', hoverinfo='name'
))

# Segment gelb: Paradeplatz -> Bellevue (2.5 Min)
fig3.add_trace(go.Scattermap(
    lat=[h1['stop_lat'], h2['stop_lat']], lon=[h1['stop_lon'], h2['stop_lon']],
    mode='lines', line=dict(color='#FFD700', width=8),
    hovertemplate='Paradeplatz → Bellevue<br>2.5 Min<extra></extra>',
    showlegend=False
))

# Segment rot: Bellevue -> Stadelhofen (5.1 Min)
fig3.add_trace(go.Scattermap(
    lat=[h2['stop_lat'], h3['stop_lat']], lon=[h2['stop_lon'], h3['stop_lon']],
    mode='lines', line=dict(color='#E53935', width=8),
    hovertemplate='Bellevue → Stadelhofen<br>5.1 Min<extra></extra>',
    showlegend=False
))

# Haltestellen
for h, name, delay in [(h1, 'Paradeplatz', 2.5), (h2, 'Bellevue', 3.8), (h3, 'Stadelhofen', 5.1)]:
    fig3.add_trace(go.Scattermap(
        lat=[h['stop_lat']], lon=[h['stop_lon']],
        mode='markers', marker=dict(size=14, color='white'),
        name=name,
        hovertemplate=f'<b>{name}</b><br>{delay} Min<extra></extra>'
    ))

fig3.update_layout(
    map=dict(style='dark', center=dict(lat=47.3695, lon=8.5430), zoom=14),
    margin=dict(l=0, r=0, t=35, b=0), height=650,
    title='Linie 11 — Verspätung auf Streckenabschnitten (synthetische Daten)',
    legend=dict(bgcolor='rgba(0,0,0,0.5)', font=dict(color='white', size=11))
)
fig3.show()
